# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for exploring the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

> The dataset covers analysis of protein abundance, modifications, and sequences in human samples, including accession, coverage, peptide counts, and molecular weights from mass spectrometry experiments.

In [ ]:
# Ensure `mlcroissant` is available
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:")
print(metadata.description)
print("---")
print("Published on:", metadata.date_published)
print("Version:", metadata.version)
print("Keywords:", getattr(metadata, 'keywords', []))
print("License:", metadata.license)


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

> All references use the `@id` identifiers as defined in the Croissant schema for consistency.

In [ ]:
# Inspect available record sets and their fields
record_sets = list(dataset.metadata.record_sets)

print("Record sets found in dataset:")
for rs in record_sets:
    print(f"  - Record set name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    Description: {rs.description}")
    print("    Fields:")
    for f in rs.fields:
        print(f"      * {f.name} (@id: {f.id}, dataType: {f.data_type})")
    print()

# For viewing the first few records in the first RecordSet
# (replace <record_set_id> with the actual '@id' from above)
if len(record_sets) != 0:
    main_record_set_id = record_sets[0].id
    print(f"First few records from record_set: {main_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        pprint.pprint(rec)
        if i >= 2:
            break
else:
    print("No record sets found in metadata.")

## 3. Data Extraction

Load data from the main record set into a DataFrame. Use record set and field `@id`s as shown above for proper mapping.

In [ ]:
# Prepare to extract data from all record sets
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display available columns for the first record set
if len(record_set_ids) != 0:
    rs0 = record_set_ids[0]
    print(f"Columns in record set {rs0}:")
    print(dataframes[rs0].columns.tolist())
    display(dataframes[rs0].head())
else:
    print("No dataframes loaded (no record sets present in schema)")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering, normalization, and grouping. We'll work with a numeric field from the first (main) record set, referencing by its `@id`.

In [ ]:
# Example: Use a numeric field such as 'cr:mw_kda' (Molecular Weight in kDa)
# Adjust field and group_fn as per your exploration needs

main_record_set = dataset.metadata.record_sets[0]
# Find common numeric fields: 'Coverage (%)', 'MW [kDa]', 'Peptide Counts', etc.
field_candidates = [(field.name, field.id) for field in main_record_set.fields if field.data_type in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Number', 'schema:Integer']]
print("Numeric field candidates:", field_candidates)

# We'll select 'MW [kDa]' if present (as an example)
field_id = None
for name, fid in field_candidates:
    if 'mw' in name.lower():
        field_id = fid
        break
if not field_id and field_candidates:
    field_id = field_candidates[0][1]

print("Chosen numeric field for filtering and normalization:", field_id)
df = dataframes[main_record_set.id]

# If the field_id is not in the dataframe columns, print and skip
if field_id not in df.columns:
    print(f"{field_id} not found in columns: {df.columns.tolist()}")
else:
    # Example threshold for MW [kDa]
    threshold = 10
    filtered_df = df[df[field_id] > threshold]
    print(f"Filtered records with {field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[field_id + '_normalized'] = (filtered_df[field_id] - filtered_df[field_id].mean()) / filtered_df[field_id].std()
    print(f"Normalized {field_id} for filtered records:")
    print(filtered_df[[field_id, field_id + '_normalized']].head())

    # Try to group by a categorical field, e.g. 'cr:accession' (uniprot id), or the first string field
    group_field = None
    for field in main_record_set.fields:
        if field.data_type == 'Text' and field.id in df.columns:
            group_field = field.id
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[field_id].mean().reset_index().rename(columns={field_id: field_id + "_mean"})
        print(f"Mean {field_id} grouped by {group_field}:")
        print(grouped_df.head())
    else:
        print("No available text group field in dataframe columns.")

## 5. Visualization

Visualize distributions and relationships between key numeric fields in the dataset.

> Fields and variable names reference their `@id` (e.g. `cr:mw_kda` for molecular weight).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of the chosen numeric field
if field_id and field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {field_id}')
    plt.xlabel(field_id)
    plt.ylabel('Count')
    plt.show()

    # If a normalized column was created, plot it as well
    norm_col = field_id + '_normalized'
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(filtered_df[norm_col].dropna(), bins=30, kde=True, color='green')
        plt.title(f'Normalized {field_id} for filtered entries')
        plt.xlabel(norm_col)
        plt.show()

    # Pairplot if another numeric field is present
    other_numeric = [f for n, f in field_candidates if f != field_id and f in df.columns]
    if other_numeric:
        plt.figure(figsize=(7,7))
        sns.scatterplot(x=df[field_id], y=df[other_numeric[0]], alpha=0.7)
        plt.xlabel(field_id)
        plt.ylabel(other_numeric[0])
        plt.title(f'{field_id} vs {other_numeric[0]}')
        plt.show()


## 6. Conclusion

- We explored the FAIR^2 Mass Spectrometry dataset using the `mlcroissant` schema.
- Examined record sets and their fields with all references by `@id` as per the Croissant standard.
- Loaded the main record set into a DataFrame and performed filtering, normalization, and simple grouping.
- Generated visualizations for main numeric fields, e.g., molecular weight distributions.

> Explore additional fields or augment this notebook for deeper biological/clinical insights using the available structured annotations.